In [102]:
# Install dependencies
%pip install anthropic python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [103]:
# Load env variable
from dotenv import load_dotenv
load_dotenv()



True

In [92]:
# Careate an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"


In [93]:
# helper functions


def add_user_message(messages, text):  
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):  
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)




def chat(messages, system=None, temperature=1.0, stop_sequences=None):

  params = {
    "model": model,
    "max_tokens": 1000,
    "messages": messages,
    "temperature": temperature
  }
  
  if system:
    params["system"] = system
  if stop_sequences:
    params["stop_sequences"] = stop_sequences


  message = client.messages.create( **params )


  return message.content[0].text

In [106]:
import json

def generate_dataset():
    prompt = """
    Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
      {
        "task": "Create a JSON configuration for an AWS Lambda function that sets up a basic python runtime with a memory allocation of 
        512 mb and a timeout of 10 seconds",
        "format": "json"
        "solution_criteria": "Key criteria for evaluating the solution"
      },
      ...additional
    ]
    

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)




In [107]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)



In [96]:

# prompt v1
prompt = """
Please provide a solution to the following task:

{task}



"""

sample_dataset = [
    {
        "task": "Create a Pyton function to extract the AWS account ID from an ARN",
        "format": "python"
    },
    {
        "task": "Write a JSON policy document that allows read-only access to a specific S3 bucket",
        "format": "json"
    },
    {
        "task": "Create a regex to validate an AWS ARN",
        "format": "regex"
    },
    
]

In [97]:
def run_prompt(test_case):
 
  prompt = f"""
  Please solve the following task:
 
  {test_case["task"]}
 * Respond only with Python, JSON, or a plain Regex
 * Do not add any comments or commentary or explanation
  """

  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "```code")
  output = chat(messages, stop_sequences=["```"])
  return output


In [111]:
import re
import ast

def grade_by_model(test_case, output):
  """ Grades the output by the model """
  # Create evaluation prompt
  eval_prompt = """
    You are an expert AWS code reviewer. Your task is to Evaluate this AI-generated solution.
    
    Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution to evaluate:
    <solution>
    {output}
    </solution>

    Criteria you should use to evaluate the solution:
    <criteria>
    {test_case["solution_criteria"]}
    </criteria>
    



    <!-- Add in the newly generated solution_criteria here -->
    "solution_criteria": "Must include runtime, memory size, timeout, and basic structure for AWS lambda configuration"

    Provide your evaluation as a structured JSON object with the following fields:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10

    Respond only with Python, JSON, or a plain regex only, nothing else. consice and direct
    Example response shape:
    {{
      "strengths": string[],
      "weaknesses": string[],
      "reasoning": string,
      "score": number
    }}

  """
  messages = []
  add_user_message(messages, eval_prompt)
  add_assistant_message(messages, "```json")
  eval_text = chat(messages, stop_sequences=["```"])
  return json.loads(eval_text)

def grade_by_human(test_case, output):
  """ Grades the output by a human """
  pass

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
        
def grade_syntax(output, test_case):

  if test_case["format"] == "python":
    return validate_python(output)
  elif test_case["format"] == "json":
    return validate_json(output)
  elif test_case["format"] == "regex":
    return validate_regex(output)


In [112]:
def run_test_case(test_case):
  """ Calls run_prompt, then grades the result """
  output = run_prompt(test_case)

  # Grade the output
  model_grade = grade_by_model(test_case, output)
  model_score = model_grade["score"]
  reasoning = model_grade["reasoning"]

  syntax_score = grade_syntax(output, test_case)

  score = (model_score + syntax_score) / 2

 

  return {
    "output" : output,
    "test_case" : test_case,
    "score" : score,
    "reasoning" : reasoning
  }



In [114]:
from statistics import mean

def run_eval(dataset):
  """ Loads the dataset and calls run_test_case with each case """
  results = []
  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")  
    
  return results



In [115]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)




Average score: 8.0
Average score: 8.0
Average score: 7.166666666666667


In [116]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef extract_bucket_name(arn):\n    match = re.search(r'arn:aws:s3:::([^/]+)', arn)\n    return match.group(1) if match else None\n",
    "test_case": {
      "task": "Write a Python function that takes an S3 bucket ARN and extracts just the bucket name from it",
      "format": "python",
      "solution_criteria": "Function should handle ARN format 'arn:aws:s3:::bucket-name' and return only 'bucket-name'. Should handle invalid inputs gracefully."
    },
    "score": 8.0,
    "reasoning": "The solution meets the basic criteria by including runtime, memory size, timeout, and basic Lambda structure. However, without seeing the actual implementation code, it's impossible to fully evaluate functionality. The configuration appears structurally correct but lacks production-grade features like proper error handling, monitoring, and security best practices."
  },
  {
    "output": "\n{\n  \"Version\": \"2012-10-17\",\n  \"Statement\": [\n    {\n      \"Effect